# Building the tables

Goes from the TheyWorkForYou debate files (Jan 2023 to Nov 2026) to `nodes.csv`, `edges.csv` and `speeches.csv`.
Split in three periods by PM: Sunak, Starmer (after the July 2024 election) and Burnham (from 20 July 2026).

In [47]:
%pip install pandas networkx matplotlib scipy

Note: you may need to restart the kernel to use updated packages.


In [48]:
import re
import json
import urllib.request
from collections import Counter
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

DEBATES_URL = "https://www.theyworkforyou.com/pwdata/scrapedxml/debates/"
YEARS = ["2023", "2024", "2025", "2026"]

PEOPLE_URL = "https://raw.githubusercontent.com/mysociety/parlparse/master/members/people.json"
PEOPLE_FILE = DATA_DIR / "people.json"

# start date of each period (PM in office)
PERIODS = {
    "sunak": "2023-01-01",
    "starmer": "2024-07-05",   # general election 4 July 2024
    "burnham": "2026-07-20",
}

### Download

Gets the debate XML files into `data/raw/`. Skips what's already there, so the first run is the slow one.

In [49]:
import time

listing = urllib.request.urlopen(DEBATES_URL).read().decode("utf-8")
files = sorted(set(re.findall(r'debates(?:' + "|".join(YEARS) + r')-\d{2}-\d{2}[a-z]\.xml', listing)))
missing = [f for f in files if not (RAW_DIR / f).exists()]
print(len(files), "files listed,", len(missing), "to download")

for f in missing:
    urllib.request.urlretrieve(DEBATES_URL + f, RAW_DIR / f)
    time.sleep(0.2)

1656 files listed, 0 to download


### Chairs

Speaker and deputies to remove. The deputies changed after the election so the list is per period.

In [50]:
CHAIRS = {
    "sunak":   {"Lindsay Hoyle", "Eleanor Laing", "Nigel Evans", "Rosie Winterton"},
    "starmer": {"Lindsay Hoyle", "Nusrat Ghani", "Judith Cummins", "Caroline Nokes"},
    "burnham": {"Lindsay Hoyle", "Nusrat Ghani", "Judith Cummins", "Caroline Nokes"},
}

def period_of(date):
    current = None
    for period, start in PERIODS.items():
        if date >= start:
            current = period
    return current

### Parties

Party of an MP at a given date, from mySociety's `people.json`. Labour/Co-op counted as Labour.

In [51]:
if not PEOPLE_FILE.exists():
    urllib.request.urlretrieve(PEOPLE_URL, PEOPLE_FILE)

people = json.loads(PEOPLE_FILE.read_text(encoding="utf-8"))
party_names = {org["id"]: org.get("name") for org in people["organizations"]}

memberships = {}
for m in people["memberships"]:
    if "person_id" in m and m.get("on_behalf_of_id"):
        memberships.setdefault(m["person_id"], []).append(m)

def party_on(person_id, date):
    for m in memberships.get(person_id, []):
        if m.get("start_date", "") <= date <= m.get("end_date", "9999-12-31"):
            party = party_names.get(m["on_behalf_of_id"], m["on_behalf_of_id"])
            return "Labour" if party == "Labour/Co-operative" else party
    return None

print(len(memberships), "people with a party membership")

4265 people with a party membership


### Speeches

Only the latest version of each day. A new heading = a new debate.

In [52]:
import html

TAG = re.compile(r'<(major-heading|minor-heading|speech)\b([^>]*)>')

def attribute(text, key):
    found = re.search(key + r'="([^"]*)"', text)
    return found.group(1) if found else None

def heading_text(content, start):
    raw = content[start:start + 400]
    raw = raw[:raw.find("</speech")] if "</speech" in raw else raw
    raw = raw[:raw.find("</major-heading")] if "</major-heading" in raw else raw
    raw = raw[:raw.find("</minor-heading")] if "</minor-heading" in raw else raw
    return " ".join(html.unescape(re.sub(r"<[^>]+>", " ", raw)).split())

rows = []
files_used = 0
for path in sorted(RAW_DIR.glob("debates*.xml")):
    content = path.read_text(encoding="utf-8", errors="replace")
    if 'latest="yes"' not in content[:8192]:
        continue
    files_used += 1
    date = re.search(r"\d{4}-\d{2}-\d{2}", path.name).group(0)
    debate_number = 0
    major, title = None, None
    for tag in TAG.finditer(content):
        kind, attrs = tag.group(1), tag.group(2)
        if kind != "speech":
            debate_number += 1          # a heading starts a new debate
            text = heading_text(content, tag.end())
            if kind == "major-heading":
                major, title = text, text
            else:
                title = text
            continue
        rows.append({
            "date": date,
            "debate": f"{path.stem}#{debate_number}",
            "major_heading": major,  # e.g. a department at Question Time
            "title": title,             # the specific debate
            "person_id": attribute(attrs, "person_id"),
            "name": attribute(attrs, "speakername"),
            "opening": heading_text(content, tag.end())[:60],  # used to spot chairing
        })

speeches = pd.DataFrame(rows)
speeches["period"] = speeches["date"].map(period_of)
print(files_used, "sitting days,", len(speeches), "speeches")
speeches.head()

560 sitting days, 240457 speeches


,date,debate,major_heading,title,person_id,name,opening,period
0,2023-01-09,debates2023-01-09b#2,Business Before Questions,New Writ,NaN,NaN,"Ordered, That Mr Speaker do issue his Warrant ...",sunak
1,2023-01-09,debates2023-01-09b#3,"Levelling Up, Housing and Communities","Levelling Up, Housing and Communities",NaN,NaN,The Secretary of State was asked—,sunak
2,2023-01-09,debates2023-01-09b#4,"Levelling Up, Housing and Communities",Potentially Unsuitable Development Proposals,uk.org.publicwhip/person/24778,Priti Patel,What steps he is taking to help communities pr...,sunak
3,2023-01-09,debates2023-01-09b#4,"Levelling Up, Housing and Communities",Potentially Unsuitable Development Proposals,uk.org.publicwhip/person/25399,Lucy Frazer,"Mr Speaker, I would like to start by apologisi...",sunak
4,2023-01-09,debates2023-01-09b#4,"Levelling Up, Housing and Communities",Potentially Unsuitable Development Proposals,uk.org.publicwhip/person/24778,Priti Patel,The Minister is well aware that communities ac...,sunak


### Cleaning

Remove the chair, MPs chairing just one debate (only in that debate), and lines with no speaker.

In [53]:
# "I call Dame X" = chairing, "I call on the Government" = not
CHAIR_LINE = r"^(Order[.,!]|I (now )?call (?!(to|on|upon|for|it|this|that|them|him|her|again)\b))"

is_chair = speeches.apply(lambda r: r["name"] in CHAIRS[r["period"]], axis=1)
no_speaker = speeches["person_id"].isna()

says_chair_line = speeches["opening"].fillna("").str.match(CHAIR_LINE) & ~is_chair & ~no_speaker
chairing = set(zip(speeches.loc[says_chair_line, "debate"], speeches.loc[says_chair_line, "person_id"]))
is_temp_chair = pd.Series([(d, p) in chairing for d, p in zip(speeches["debate"], speeches["person_id"])], index=speeches.index)

print("Speaker / Deputy Speaker speeches removed:", is_chair.sum())
print("temporary chair speeches removed:", is_temp_chair.sum())
print("  by:", speeches.loc[is_temp_chair, "name"].value_counts().head(8).to_dict())
print("speeches with no speaker removed:", no_speaker.sum())

speeches = speeches[~is_chair & ~is_temp_chair & ~no_speaker].reset_index(drop=True)
print("speeches kept:", len(speeches))

Speaker / Deputy Speaker speeches removed: 17110
temporary chair speeches removed: 652
  by: {'Roger Gale': 535, 'Christopher Chope': 41, 'Edward Leigh': 40, 'Siobhain McDonagh': 29, 'Clive Betts': 5, 'Jim Shannon': 1, 'Peter Gibson': 1}
speeches with no speaker removed: 11557
speeches kept: 211138


### Edges

A -> B gets +1 every time B speaks right after A, inside the same debate. No self-loops.

In [54]:
pairs = Counter()
for (period, debate), group in speeches.groupby(["period", "debate"], sort=False):
    ids = group["person_id"].tolist()
    for a, b in zip(ids, ids[1:]):
        if a != b:
            pairs[(a, b, period)] += 1

edges = pd.DataFrame(
    [(a, b, p, w) for (a, b, p), w in pairs.items()],
    columns=["source", "target", "period", "weight"],
)
edges.groupby("period").agg(edges=("weight", "size"), exchanges=("weight", "sum"))

,edges,exchanges
period,,
burnham,8100,10119
starmer,59528,120422
sunak,35739,67796


### Nodes

In [55]:
in_edges = set(edges["source"]) | set(edges["target"])
speeches = speeches[speeches["person_id"].isin(in_edges)]

last = speeches.sort_values("date").groupby("person_id").last()
nodes = pd.DataFrame({
    "id": last.index,
    "name": last["name"].values,
    "party": [party_on(pid, d) for pid, d in zip(last.index, last["date"])],
})

# party + number of speeches per period (a few MPs switched party)
for period in PERIODS:
    in_period = speeches[speeches["period"] == period].sort_values("date").groupby("person_id").last()
    party_then = {pid: party_on(pid, d) for pid, d in zip(in_period.index, in_period["date"])}
    nodes[f"party_{period}"] = nodes["id"].map(party_then)
    nodes[f"speeches_{period}"] = nodes["id"].map(speeches[speeches["period"] == period].groupby("person_id").size()).fillna(0).astype(int)

print(len(nodes), "MPs,", nodes["party"].isna().sum(), "without a party found")
nodes["party"].value_counts()

994 MPs, 0 without a party found


party
Labour                                443
Conservative                          369
Liberal Democrat                       72
Scottish National Party                47
Independent                            24
DUP                                     8
Reform UK                               8
Green                                   6
Plaid Cymru                             5
Alliance                                2
Alba                                    2
Social Democratic and Labour Party      2
Workers Party                           1
Reclaim                                 1
Traditional Unionist Voice              1
UUP                                     1
Your Party                              1
Restore Britain                         1
Name: count, dtype: int64

### Save

In [56]:
nodes.to_csv(DATA_DIR / "nodes.csv", index=False)
edges.to_csv(DATA_DIR / "edges.csv", index=False)
speeches[["date", "period", "debate", "major_heading", "title", "person_id"]].to_csv(DATA_DIR / "speeches.csv", index=False)
print("saved nodes.csv, edges.csv and speeches.csv in", DATA_DIR)

saved nodes.csv, edges.csv and speeches.csv in data


Quick check: strongest edges in each period

In [57]:
names = dict(zip(nodes["id"], nodes["name"]))
check = edges.assign(source_name=edges["source"].map(names), target_name=edges["target"].map(names))
for period in PERIODS:
    print(f"--- {period}")
    print(check[check["period"] == period].nlargest(10, "weight")[["source_name", "target_name", "weight"]].to_string(index=False))

--- sunak
       source_name    target_name  weight
      Keir Starmer    Rishi Sunak     271
       Rishi Sunak   Keir Starmer     268
       Rishi Sunak  Stephen Flynn      92
     Stephen Flynn    Rishi Sunak      91
       Lucy Powell Penny Mordaunt      53
Thangam Debbonaire Penny Mordaunt      46
    Penny Mordaunt   Deidre Brock      44
    Penny Mordaunt     Ian Mearns      44
    Penny Mordaunt    Jim Shannon      44
       Jim Shannon Penny Mordaunt      44
--- starmer
    source_name     target_name  weight
   Keir Starmer   Kemi Badenoch     361
  Kemi Badenoch    Keir Starmer     360
   Keir Starmer    Edward Davey     141
   Edward Davey    Keir Starmer     140
    John Healey James Cartlidge      59
James Cartlidge     John Healey      59
   Jesse Norman     Lucy Powell      58
   Jesse Norman   Alan Campbell      56
  Alex Burghart     Hilary Benn      47
    Hilary Benn   Alex Burghart      46
--- burnham
   source_name    target_name  weight
  Andy Burnham  Kemi Baden

MPs with no party found (should be empty)

In [58]:
nodes[nodes["party"].isna()]

,id,name,party,party_sunak,speeches_sunak,party_starmer,speeches_starmer,party_burnham,speeches_burnham
